# alignment

> Alignment service for temporal coordination via Silero VAD plugin

In [ ]:
#| default_exp services.alignment

In [ ]:
#| export
from typing import List, Dict, Any, Optional
import asyncio

from cjm_plugin_system.core.manager import PluginManager

from cjm_fasthtml_workflow_transcript_decomp.core.models import VADChunk, WorkingSegment

## AlignmentService

This service wraps the Silero VAD plugin to provide voice activity detection data. It processes audio files to extract time ranges where speech is detected, which are then used to align text segments with their corresponding audio positions.

In [ ]:
#| export
class AlignmentService:
    """Service for temporal alignment via Silero VAD plugin."""
    
    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager for accessing VAD plugin
        plugin_name: str = "cjm-media-plugin-silero-vad"  # Name of the VAD plugin
    ):
        """Initialize the alignment service."""
        self._manager = plugin_manager
        self._plugin_name = plugin_name
    
    def is_available(self) -> bool:  # True if plugin is loaded and ready
        """Check if the VAD plugin is available."""
        return self._manager.get_plugin(self._plugin_name) is not None
    
    def ensure_loaded(
        self,
        config: Optional[Dict[str, Any]] = None  # Optional plugin configuration
    ) -> bool:  # True if successfully loaded
        """Ensure the VAD plugin is loaded."""
        if self.is_available():
            return True
        
        # Try to find and load the plugin
        meta = self._manager.get_discovered_meta(self._plugin_name)
        if meta:
            return self._manager.load_plugin(meta, config or {"threshold": 0.5})
        return False
    
    async def analyze_audio_async(
        self,
        media_path: str  # Path to audio/video file
    ) -> tuple[List[VADChunk], float]:  # (VAD chunks, total duration)
        """Analyze audio file and return VAD chunks."""
        if not self.is_available():
            raise RuntimeError(f"Plugin {self._plugin_name} not loaded")
        
        # Execute plugin
        result = await self._manager.execute_plugin_async(
            self._plugin_name,
            media_path=media_path
        )
        
        # Extract ranges and metadata
        ranges = result.get('ranges', [])
        metadata = result.get('metadata', {})
        duration = metadata.get('duration', 0.0)
        
        # Convert to VADChunk objects
        chunks = []
        for idx, r in enumerate(ranges):
            chunk = VADChunk(
                index=idx,
                start_time=r.get('start', r.get('start_time', 0.0)),
                end_time=r.get('end', r.get('end_time', 0.0))
            )
            chunks.append(chunk)
        
        return chunks, duration
    
    def analyze_audio(
        self,
        media_path: str  # Path to audio/video file
    ) -> tuple[List[VADChunk], float]:  # (VAD chunks, total duration)
        """Analyze audio file synchronously."""
        return asyncio.get_event_loop().run_until_complete(
            self.analyze_audio_async(media_path)
        )

## Alignment Helpers

These functions support the UI operations for linking VAD chunks to text segments.

In [ ]:
#| export
def assign_chunk_to_segment(
    segment: WorkingSegment,  # Segment to assign chunk to
    chunk: VADChunk  # VAD chunk to assign
) -> tuple[WorkingSegment, VADChunk]:  # Updated segment and chunk
    """Assign a VAD chunk to a segment, updating time boundaries."""
    # Add chunk index to segment
    if chunk.index not in segment.vad_chunk_indices:
        segment.vad_chunk_indices.append(chunk.index)
        segment.vad_chunk_indices.sort()
    
    # Update segment time boundaries
    if segment.start_time is None or chunk.start_time < segment.start_time:
        segment.start_time = chunk.start_time
    if segment.end_time is None or chunk.end_time > segment.end_time:
        segment.end_time = chunk.end_time
    
    # Mark chunk as assigned
    chunk.assigned_segment = segment.index
    
    return segment, chunk

In [ ]:
#| export
def unassign_chunk_from_segment(
    segment: WorkingSegment,  # Segment to remove chunk from
    chunk: VADChunk,  # VAD chunk to unassign
    all_chunks: List[VADChunk]  # All chunks for recalculating times
) -> tuple[WorkingSegment, VADChunk]:  # Updated segment and chunk
    """Remove a VAD chunk assignment from a segment."""
    # Remove chunk index from segment
    if chunk.index in segment.vad_chunk_indices:
        segment.vad_chunk_indices.remove(chunk.index)
    
    # Clear chunk assignment
    chunk.assigned_segment = None
    
    # Recalculate segment time boundaries from remaining chunks
    if segment.vad_chunk_indices:
        assigned_chunks = [c for c in all_chunks if c.index in segment.vad_chunk_indices]
        segment.start_time = min(c.start_time for c in assigned_chunks)
        segment.end_time = max(c.end_time for c in assigned_chunks)
    else:
        segment.start_time = None
        segment.end_time = None
    
    return segment, chunk

In [ ]:
#| export
def auto_align_sequential(
    segments: List[WorkingSegment],  # Segments to align
    chunks: List[VADChunk]  # VAD chunks to distribute
) -> tuple[List[WorkingSegment], List[VADChunk]]:  # Updated segments and chunks
    """Automatically align chunks to segments sequentially."""
    if not segments or not chunks:
        return segments, chunks
    
    # Simple proportional distribution based on text length
    total_text_len = sum(len(s.text) for s in segments)
    if total_text_len == 0:
        return segments, chunks
    
    chunk_idx = 0
    total_chunks = len(chunks)
    
    for segment in segments:
        # Calculate proportion of chunks for this segment
        proportion = len(segment.text) / total_text_len
        num_chunks = max(1, round(proportion * total_chunks))
        
        # Assign chunks to this segment
        end_idx = min(chunk_idx + num_chunks, total_chunks)
        for i in range(chunk_idx, end_idx):
            assign_chunk_to_segment(segment, chunks[i])
        
        chunk_idx = end_idx
        if chunk_idx >= total_chunks:
            break
    
    # Assign any remaining chunks to last segment
    if chunk_idx < total_chunks and segments:
        for i in range(chunk_idx, total_chunks):
            assign_chunk_to_segment(segments[-1], chunks[i])
    
    return segments, chunks

In [ ]:
#| export
def get_unassigned_chunks(
    chunks: List[VADChunk]  # All VAD chunks
) -> List[VADChunk]:  # Chunks without segment assignments
    """Get all VAD chunks that are not assigned to any segment."""
    return [c for c in chunks if c.assigned_segment is None]

In [ ]:
#| export
def get_segments_without_time(
    segments: List[WorkingSegment]  # All segments
) -> List[WorkingSegment]:  # Segments missing time alignment
    """Get all segments that don't have time alignment."""
    return [s for s in segments if s.start_time is None or s.end_time is None]

## Tests

The following cells demonstrate the alignment service and helper functions.

In [ ]:
# Test assign_chunk_to_segment
segment = WorkingSegment(index=0, text="The art of war is vital.")
chunk = VADChunk(index=0, start_time=0.0, end_time=2.5)

print("Before assignment:")
print(f"  Segment times: {segment.start_time} - {segment.end_time}")
print(f"  Chunk assigned: {chunk.assigned_segment}")

segment, chunk = assign_chunk_to_segment(segment, chunk)

print("\nAfter assignment:")
print(f"  Segment times: {segment.start_time} - {segment.end_time}")
print(f"  Segment VAD indices: {segment.vad_chunk_indices}")
print(f"  Chunk assigned to: {chunk.assigned_segment}")

Before assignment:
  Segment times: None - None
  Chunk assigned: None

After assignment:
  Segment times: 0.0 - 2.5
  Segment VAD indices: [0]
  Chunk assigned to: 0


In [ ]:
# Test unassign_chunk_from_segment
all_chunks = [
    VADChunk(index=0, start_time=0.0, end_time=2.5, assigned_segment=0),
    VADChunk(index=1, start_time=2.5, end_time=5.0, assigned_segment=0),
    VADChunk(index=2, start_time=5.0, end_time=7.5)
]

segment = WorkingSegment(
    index=0,
    text="Test segment",
    start_time=0.0,
    end_time=5.0,
    vad_chunk_indices=[0, 1]
)

print("Before unassign:")
print(f"  Segment times: {segment.start_time} - {segment.end_time}")
print(f"  Segment VAD indices: {segment.vad_chunk_indices}")

# Unassign chunk 1
segment, chunk = unassign_chunk_from_segment(segment, all_chunks[1], all_chunks)

print("\nAfter unassigning chunk 1:")
print(f"  Segment times: {segment.start_time} - {segment.end_time}")
print(f"  Segment VAD indices: {segment.vad_chunk_indices}")
print(f"  Chunk 1 assigned: {all_chunks[1].assigned_segment}")

Before unassign:
  Segment times: 0.0 - 5.0
  Segment VAD indices: [0, 1]

After unassigning chunk 1:
  Segment times: 0.0 - 2.5
  Segment VAD indices: [0]
  Chunk 1 assigned: None


In [ ]:
# Test auto_align_sequential
segments = [
    WorkingSegment(index=0, text="Short."),
    WorkingSegment(index=1, text="This is a medium length sentence with more words."),
    WorkingSegment(index=2, text="Tiny.")
]

chunks = [VADChunk(index=i, start_time=i*1.0, end_time=(i+1)*1.0) for i in range(6)]

print("Before auto-alignment:")
for s in segments:
    print(f"  Segment {s.index}: '{s.text[:20]}...' has {len(s.vad_chunk_indices)} chunks")

auto_align_sequential(segments, chunks)

print("\nAfter auto-alignment:")
for s in segments:
    print(f"  Segment {s.index}: '{s.text[:20]}...' has chunks {s.vad_chunk_indices}, time {s.start_time}-{s.end_time}")

Before auto-alignment:
  Segment 0: 'Short....' has 0 chunks
  Segment 1: 'This is a medium len...' has 0 chunks
  Segment 2: 'Tiny....' has 0 chunks

After auto-alignment:
  Segment 0: 'Short....' has chunks [0], time 0.0-1.0
  Segment 1: 'This is a medium len...' has chunks [1, 2, 3, 4, 5], time 1.0-6.0
  Segment 2: 'Tiny....' has chunks [], time None-None


In [ ]:
# Test get_unassigned_chunks and get_segments_without_time
chunks = [
    VADChunk(index=0, start_time=0.0, end_time=1.0, assigned_segment=0),
    VADChunk(index=1, start_time=1.0, end_time=2.0),  # Unassigned
    VADChunk(index=2, start_time=2.0, end_time=3.0, assigned_segment=1),
    VADChunk(index=3, start_time=3.0, end_time=4.0)   # Unassigned
]

segments = [
    WorkingSegment(index=0, text="Has time", start_time=0.0, end_time=1.0),
    WorkingSegment(index=1, text="No time"),  # Missing time alignment
    WorkingSegment(index=2, text="Also has time", start_time=2.0, end_time=3.0)
]

unassigned = get_unassigned_chunks(chunks)
print(f"Unassigned chunks: {[c.index for c in unassigned]}")

no_time = get_segments_without_time(segments)
print(f"Segments without time: {[s.index for s in no_time]}")

Unassigned chunks: [1, 3]
Segments without time: [1]


### AlignmentService with Silero VAD Plugin

These tests require the Silero VAD plugin to be installed and discoverable.

In [ ]:
#| eval: false
# Test AlignmentService with Silero VAD plugin
from pathlib import Path
from cjm_plugin_system.core.manager import PluginManager

# Calculate project root from notebook location (nbs/services/ -> project root)
project_root = Path.cwd().parent.parent
manifests_dir = project_root / ".cjm" / "manifests"

# Create plugin manager with explicit search path
manager = PluginManager(search_paths=[manifests_dir])
manager.discover_manifests()

print(f"Discovered {len(manager.discovered)} plugins from {manifests_dir}")

# Check if VAD plugin is available
vad_meta = manager.get_discovered_meta("cjm-media-plugin-silero-vad")
if vad_meta:
    print(f"Found plugin: {vad_meta.name} v{vad_meta.version}")
else:
    print("Silero VAD plugin not found - install via plugins.yaml")

[PluginManager] Discovered manifest: cjm-transcription-plugin-voxtral-hf from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests/cjm-transcription-plugin-voxtral-hf.json
[PluginManager] Discovered manifest: cjm-system-monitor-nvidia from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests/cjm-system-monitor-nvidia.json
[PluginManager] Discovered manifest: cjm-transcription-plugin-whisper from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests/cjm-transcription-plugin-whisper.json
[PluginManager] Discovered manifest: cjm-media-plugin-silero-vad from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests/cjm-media-plugin-silero-vad.json
[PluginManager] Discovered manifest: cjm-graph-plugin-sqlite from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manif

Discovered 6 plugins from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests
Found plugin: cjm-media-plugin-silero-vad v0.0.1


In [ ]:
#| eval: false
# Initialize and test AlignmentService
if vad_meta:
    # Load the plugin
    manager.load_plugin(vad_meta, {"threshold": 0.5})
    
    align_service = AlignmentService(manager)
    print(f"Plugin available: {align_service.is_available()}")
    
    # Test audio analysis (use await directly - Jupyter supports top-level await)
    audio_file = "/mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcription-plugin-voxtral-hf/test_files/02 - 1. Laying Plans.mp3"
    
    chunks, duration = await align_service.analyze_audio_async(audio_file)
    
    print(f"\nAudio duration: {duration:.2f}s")
    print(f"Detected {len(chunks)} VAD chunks")
    print("\nFirst 5 chunks:")
    for chunk in chunks[:5]:
        print(f"  [{chunk.index}] {chunk.start_time:.2f}s - {chunk.end_time:.2f}s (duration: {chunk.duration:.2f}s)")

[PluginManager] Launching worker for cjm-media-plugin-silero-vad...


[cjm-media-plugin-silero-vad] Starting worker on port 43703...
[cjm-media-plugin-silero-vad] Logs: /home/innom-dt/.cjm/logs/cjm-media-plugin-silero-vad.log


[PluginManager] HTTP Request: GET http://127.0.0.1:43703/health "HTTP/1.1 200 OK"
[PluginManager] HTTP Request: POST http://127.0.0.1:43703/initialize "HTTP/1.1 200 OK"
[PluginManager] Loaded plugin: cjm-media-plugin-silero-vad
[PluginManager] HTTP Request: POST http://127.0.0.1:43703/execute "HTTP/1.1 200 OK"


[cjm-media-plugin-silero-vad] Worker ready.
Plugin available: True

Audio duration: 256.39s
Detected 89 VAD chunks

First 5 chunks:
  [0] 1.10s - 2.30s (duration: 1.20s)
  [1] 4.80s - 5.90s (duration: 1.10s)
  [2] 6.60s - 9.80s (duration: 3.20s)
  [3] 10.70s - 12.40s (duration: 1.70s)
  [4] 12.60s - 15.40s (duration: 2.80s)


In [ ]:
#| eval: false
# Cleanup
if vad_meta:
    manager.unload_all()
    print("Plugins unloaded")

[PluginManager] HTTP Request: POST http://127.0.0.1:43703/cleanup "HTTP/1.1 200 OK"
[PluginManager] Unloaded plugin: cjm-media-plugin-silero-vad


Plugins unloaded


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()